# Super-resolution for CLSM: eSRRF, ISM, SOFISM and s²ISM

Every reconstruction in this notebook lives on `tttrlib.CLSMSuperRes`. They sharpen an image for genuinely different reasons, and knowing which is which decides what you can claim from a result:

| Method | Exploits | Needs | Linear? |
|---|---|---|---|
| **eSRRF** | the *image* — gradients converge on an emitter | nothing else | no, can create structure |
| **ISM (APR)** | the *detector* — each element sees a displaced PSF | an array detector | yes, photon-conserving |
| **SOFISM** | the *emitters* — independent blinking | array **and** fluctuations | no, a cumulant |
| **s²ISM** | a *PSF model* — joint inversion over axial planes | array and a PSF | no, iterative ML |

References: eSRRF — [Laine *et al.*, Nat. Methods 20, 1949 (2023)](https://doi.org/10.1038/s41592-023-02057-w), ported from [NanoJ-eSRRF](https://github.com/HenriquesLab/NanoJ-eSRRF). ISM and focus-ISM — [Tortarolo *et al.*, Nat. Commun. 13, 7929 (2022)](https://doi.org/10.1038/s41467-022-35333-y), ported from [BrightEyes-ISM](https://github.com/VicidominiLab/BrightEyes-ISM). SOFISM — [Sroda *et al.*, Optica 7, 1308 (2020)](https://doi.org/10.1364/OPTICA.399600). s²ISM — [Zunino *et al.*, Nat. Photonics (2025)](https://doi.org/10.1038/s41566-025-01695-0).

The narrative guide is [Super-resolution: eSRRF, ISM, SOFISM and s²ISM](../superres-guide.rst).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import fftconvolve
import tttrlib

# The tubulin phantom and the array-detector PSF model are simulation helpers
# shipped with the examples, not part of the library.
repo = Path.cwd().parent.parent
for extra in (repo / "examples" / "simulation",):
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from generate_tubulin_phantom import generate_tubulin_phantom
from simulate import generate_ism_psf

print("1. Simulating Ground Truth Tubulin Network Phantom (25 nm/pixel)...")
ground_truth = generate_tubulin_phantom(n_filaments=12, size_px=128, pixel_size_nm=25.0)

print("2. Simulating Physical 5x5 SPAD Array Detector PSFs (0.5 AU pitch)...")
psf_sim = generate_ism_psf(
    na=1.4,
    wavelength_exc=488.0,
    wavelength_det=520.0,
    n_det=5,            # 5x5 SPAD detector array
    pitch_au=0.5,       # 0.5 Airy units pitch
    nx=64, ny=64,
    pixel_size_nm=25.0
)
channel_psfs = psf_sim['channel_psfs']         # Shape (25, 64, 64)
detector_offsets = psf_sim['detector_offsets'] # Shape (25, 2)

print("3. Convolving Phantom with SPAD Detector Array PSFs...")
n_det, ny, nx = 25, ground_truth.shape[0], ground_truth.shape[1]
spad_cube = np.zeros((n_det, ny, nx), dtype=np.float64)
for k in range(n_det):
    spad_cube[k] = fftconvolve(ground_truth, channel_psfs[k], mode='same')

# Add Poisson photon noise
rng = np.random.default_rng(1)
spad_cube = rng.poisson(np.clip(spad_cube * 150.0, 0, None)).astype(np.float64)
print(f"Simulated SPAD Array Cube Shape: {spad_cube.shape}")

In [ ]:
print("4. Performing ISM Super-Resolution Reconstructions from Simulation...")
# Standard CLSM Open-Pinhole Sum Image
clsm_sum = spad_cube.sum(axis=0)

# The shift vectors APR is built on: how far each detector element's image has
# to move to register onto the central element. They should trace out the
# detector lattice, scaled by roughly one half.
shifts = tttrlib.CLSMSuperRes.shift_vectors(spad_cube, usf=10)
print(f"Shift vectors (dy, dx), first three elements:\n{shifts[:3]}")

# Adaptive Pixel Reassignment (APR-ISM)
apr_ism = tttrlib.CLSMSuperRes.apr_reconstruction(spad_cube, usf=10)[0]

# Focus-ISM: in-focus signal, out-of-focus background, and the APR sum
focus_signal, focus_background, _ = tttrlib.CLSMSuperRes.focus_reconstruction(
    spad_cube, sigma_bound=2.0, calibration_size=16, parallelize=True
)

print("Reconstructions complete.")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

panels = [
    ('A) Ground truth (tubulin phantom)', ground_truth),
    ('B) Confocal, open pinhole (channel sum)', clsm_sum),
    ('C) APR-ISM (adaptive pixel reassignment)', apr_ism),
    ('D) Focus-ISM: in-focus signal', focus_signal),
    ('E) Focus-ISM: out-of-focus background', focus_background),
    ('F) eSRRF map of the APR-ISM image',
     tttrlib.CLSMSuperRes.rgc_map(apr_ism, magnification=4, fwhm=2.5,
                                  sensitivity=1, intensity_weighting=True)),
]
for ax, (title, img) in zip(axes.ravel(), panels):
    ax.imshow(img, cmap='magma', origin='lower')
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

## SOFISM

SOFISM multiplies the ISM and SOFI mechanisms. At each scan position the array records a short time series, and for each *pair* of elements the cross-correlation of the fluctuations is formed:

$$C_{ij}(\mathbf{r},\tau) = \frac{1}{N_t-\tau}\sum_t \delta I_i(\mathbf{r},t)\,\delta I_j(\mathbf{r},t+\tau)$$

Its effective PSF is the **product** of the two elements' PSFs. The pair behaves as one virtual detector midway between them, so it is reassigned by $\vec{v}_{ij}=(\vec{v}_i+\vec{v}_j)/2$ and the shifted images are summed. Autocorrelation terms are left out — their shot noise does not cancel.

The contrast depends on the emitters blinking **independently**, so the cell below also runs the negative control: the same object held permanently on. That must produce nothing, and it is the check worth repeating on real data.

In [ ]:
# A blinking version of the same object, recorded as a time series per scan
# position. Only 3x3 elements and a small field, to keep the notebook quick.
N_TIME, P_ON, SIDE_S = 24, 0.3, 3
small = ground_truth[32:96, 32:96]
psf_small = generate_ism_psf(na=1.4, wavelength_exc=488.0, wavelength_det=520.0,
                             n_det=SIDE_S, pitch_au=0.5, nx=48, ny=48,
                             pixel_size_nm=25.0)["channel_psfs"]
n_det_s, ny_s, nx_s = psf_small.shape[0], *small.shape

ys, xs = np.nonzero(small > 0.05 * small.max())
amp = small[ys, xs]

def acquire(blink, seed):
    rng = np.random.default_rng(seed)
    cube = np.zeros((N_TIME, n_det_s, ny_s, nx_s))
    for t in range(N_TIME):
        on = rng.random(len(amp)) < P_ON if blink else np.ones(len(amp), bool)
        frame = np.zeros((ny_s, nx_s)); frame[ys[on], xs[on]] = amp[on]
        for k in range(n_det_s):
            cube[t, k] = fftconvolve(frame, psf_small[k], mode="same")
    return rng.poisson(np.clip(cube * (60.0 / P_ON), 0, None)).astype(float)

blinking = acquire(True, 2)
static = acquire(False, 3)

sofism = tttrlib.CLSMSuperRes.sofism_reconstruction(blinking, lag=0, usf=8)
sofism_static = tttrlib.CLSMSuperRes.sofism_reconstruction(static, lag=0, usf=8)
confocal_s = blinking.mean(axis=0).sum(axis=0)
apr_s = tttrlib.CLSMSuperRes.apr_reconstruction(blinking.mean(axis=0), usf=8)[0]

print(f"peak SOFISM  blinking {np.abs(sofism).max():.4g}"
      f"   static {np.abs(sofism_static).max():.4g}"
      f"   ratio {np.abs(sofism).max() / max(np.abs(sofism_static).max(), 1e-12):.0f}x")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
vmax = np.clip(sofism, 0, None).max()
for ax, (title, img, vm) in zip(axes, [
    ("Confocal (channel sum)", confocal_s, None),
    ("APR-ISM", apr_s, None),
    ("SOFISM", np.clip(sofism, 0, None), None),
    ("SOFISM, non-blinking control\n(same colour scale)",
     np.clip(sofism_static, 0, None), vmax),
]):
    ax.imshow(img, cmap="magma", origin="lower", vmin=0, vmax=vm)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.tight_layout(); plt.show()

## s²ISM

s²ISM is **not** a reassignment. It treats the array as $N_{ch}$ images of one object seen through $N_{ch}$ different PSFs and inverts them jointly by multi-image Richardson–Lucy over a stack of axial planes:

$$\hat{I}_{ch} = \sum_z O_z * h_{z,ch}, \qquad O_z \leftarrow O_z \cdot \sum_{ch}\frac{I_{ch}}{\hat{I}_{ch}} \star h_{z,ch}$$

The **sectioning** comes from the axial stack: out-of-focus haze is explained by the defocused planes instead of being smeared into the focal one, which a single-plane deconvolution cannot do because it has nowhere else to put it. Unlike APR it needs a PSF *model* — it does not derive one from the data.

In [ ]:
# sharp beads in focus, under a broad out-of-focus halo
NS, SIDE_Z, NZ = 64, 3, 3
yy, xx = np.mgrid[0:NS, 0:NS]
psf_z = np.zeros((NZ, SIDE_Z * SIDE_Z, NS, NS))
for z in range(NZ):
    sigma = 1.6 + 3.0 * abs(z - NZ // 2)          # sharp in focus, broad away
    for row in range(SIDE_Z):
        for col in range(SIDE_Z):
            k = row * SIDE_Z + col
            dx = (col - (SIDE_Z - 1) / 2) * 1.6
            dy = (row - (SIDE_Z - 1) / 2) * 1.6
            psf_z[z, k] = np.exp(-(((xx - (NS / 2 + dx / 2)) ** 2
                                    + (yy - (NS / 2 + dy / 2)) ** 2) / (2 * sigma ** 2)))

focal_obj = np.zeros((NS, NS))
for cy, cx in [(28, 26), (28, 34), (38, 30)]:
    focal_obj[cy, cx] = 1.0
haze = np.exp(-(((xx - 30) ** 2 + (yy - 34) ** 2) / (2 * 11.0 ** 2))) * 0.02

rng = np.random.default_rng(4)
cube_z = np.stack([
    fftconvolve(focal_obj, psf_z[NZ // 2, k], mode="same")
    + fftconvolve(haze, psf_z[0, k], mode="same")
    for k in range(SIDE_Z * SIDE_Z)])
cube_z = rng.poisson(np.clip(cube_z, 0, None) * 4000.0).astype(float)

obj = tttrlib.CLSMSuperRes.s2ism_reconstruction(cube_z, psf_z, max_iter=60)
focal_plane, defocused = obj[NZ // 2], obj.sum(axis=0) - obj[NZ // 2]
conf_z = cube_z.sum(axis=0)

print(f"photons in the focal plane : {100 * focal_plane.sum() / obj.sum():.1f} %")
print(f"contrast, confocal -> s2ISM: {conf_z.max() / np.median(conf_z):.1f}"
      f" -> {focal_plane.max() / max(np.median(focal_plane), 1e-9):.1f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, (title, img) in zip(axes, [
    ("Confocal (channel sum)", conf_z),
    ("APR-ISM", tttrlib.CLSMSuperRes.apr_reconstruction(cube_z, usf=8)[0]),
    ("s2ISM, focal plane", focal_plane),
    ("s2ISM, defocused planes", defocused),
]):
    ax.imshow(np.clip(img, 0, None), cmap="magma", origin="lower")
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.tight_layout(); plt.show()

## Choosing a method

* **No array detector?** Only eSRRF applies.
* **A gain you can defend without qualification?** APR — linear and photon-conserving, but bounded near $\sqrt{2}$.
* **Sample blinks, and you can time-resolve it within the pixel dwell?** SOFISM reaches about $2\times$, and from a physical mechanism rather than a rendering choice.
* **Out-of-focus background is the problem?** s²ISM for true sectioning if you have a PSF model, focus-ISM for a cheaper two-component split.

Two cautions worth carrying away:

1. **eSRRF is nonlinear.** On a target of shrinking line pairs its modulation is not monotonic — it recovers contrast at separations where the pairs have already merged, by splitting a merged blur into an artificial doublet. Trust a resolution claim only where every wider feature is also resolved.
2. **eSRRF on an APR image** is sound where the background is not photon-starved, but the non-negativity clip after registration rectifies the interpolation's negative lobes. In dim regions that leaves a positively biased, spatially *correlated* background, which is exactly what a gradient-convergence detector turns into spurious puncta. On sparse structure over a dark background, prefer SOFISM or plain APR.